# Workshop Notebook: From JSON Chat to Raw LLM Prompt

This notebook demonstrates what *actually* reaches an LLM.

Modern SDKs send JSON, but models only receive formatted **text prompts**.

We will show:

1. JSON chat input
2. Conversion into a prompt template
3. Adding tools into prompts
4. Sending raw prompts to Ollama
5. Viewing raw model output


## Step 1 — Chat messages in JSON

In [12]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What color is the sky at night?"}
]

messages

[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user', 'content': 'What color is the sky at night?'}]

## Step 2 — Tools provided by the application

Tools are also injected as text into the prompt.

In [13]:
tools = [
    {
        "name": "get_weather",
        "description": "Get weather for a city",
        "parameters": {"city": "string"}
    }
]

tools

[{'name': 'get_weather',
  'description': 'Get weather for a city',
  'parameters': {'city': 'string'}}]

## Step 3 — Build prompt template

Server converts JSON messages + tools into plain text.

In [14]:
def build_prompt(messages, tools=None):
    role_tokens = {
        "system": "<|system|>",
        "user": "<|user|>",
        "assistant": "<|assistant|>",
        "tool": "<|tool|>"
    }

    parts = []

    if tools:
        tool_lines = ["Available tools:"]
        for t in tools:
            tool_lines.append(f"Tool: {t['name']}")
            tool_lines.append(f"Description: {t['description']}")

        parts.append("<|system|>\n" + "\n".join(tool_lines))

    for msg in messages:
        token = role_tokens[msg['role']]
        parts.append(f"{token}\n{msg['content']}")

    # assistant continuation marker
    parts.append("<|assistant|>\n")

    return "\n".join(parts)

prompt = build_prompt(messages, tools)
print(prompt)

<|system|>
Available tools:
Tool: get_weather
Description: Get weather for a city
<|system|>
You are a helpful assistant.
<|user|>
What color is the sky at night?
<|assistant|>



This printed text is exactly what gets sent to the LLM.

Everything else (SDK, JSON, tool metadata) happens outside the model.

## Step 4 — Send raw prompt to Ollama

This sends the prompt without chat wrappers.

Make sure Ollama is running locally.

In [15]:
from mlx_lm import load, generate

model, tokenizer = load("mlx-community/Phi-3-mini-4k-instruct-4bit")

prompt = """<|system|>
You can call tools.

Tool: get_weather
Description: Get weather for a city.

Return tool calls as JSON.

<|user|>
Weather in Paris?

<|assistant|>
"""


out = generate(model, tokenizer, prompt, max_tokens=80)
print(out)


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 160632.92it/s]


To get the weather in Paris, I would use the `get_weather` tool. Here's how the JSON output might look:

```json
{
  "city": "Paris",
  "temperature": "15°C",
dependent on the time of day
  "condition": "Partly cloudy",
  "humidity


In [10]:
print("PROMPT SENT TO MODEL:\n")
print(prompt)

print("\nMODEL OUTPUT:\n")
print(generate(model, tokenizer, prompt, max_tokens=80))

PROMPT SENT TO MODEL:

<|user|>
What color is the sky?
<|assistant|>


MODEL OUTPUT:

Blue.

This question is straightforward and has a fixed answer. The color of the sky is typically perceived as blue due to the scattering of sunlight by the atmosphere, a phenomenon known as Rayleigh scattering. The shorter blue wavelengths are scattered more than other colors, which is why we see a blue sky most of the temp.


What is


## Final takeaway

LLMs do not receive JSON.

They only receive formatted text prompts, and predict what text comes next.

Chat systems, tools, and agents are prompt engineering layers around text completion.